# 第 1 周末练习 —— 系统设计面试教练

## 练习目标（理念）

为了展示你对 **OpenAI API**（本练习经 OpenRouter）以及本地 **Ollama** 的熟悉程度，请构建一个小工具：

- **输入**：一道系统设计题（例如「设计像 bit.ly 的短链服务」）
- **输出**：结构化、偏面试风格的讲解（需求澄清 → 高层架构 → 深挖 → 扩展瓶颈）
- **额外要求**：默认 **流式（streaming）** 一边生成一边用 Markdown 刷新显示

这是你在课程期间自己也能天天用的工具：准备系统设计面试时，丢题目进来练。

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions API | `client.chat.completions.create(...)` |
| `messages`（system / user） | system 定「面试官角色」，user 放具体题目 |
| 流式输出 `stream=True` | 逐块拼接，用 `update_display` 刷新 Markdown |
| 云端小模型 | `gpt-4o-mini`（常量 `MODEL_GPT`，经 OpenRouter） |
| Ollama 本地模型 | `llama3.2`（常量 `MODEL_LLAMA`），OpenAI 兼容 `/v1` |

## 怎么跑

1. 从上到下依次运行每个单元格（Shift+Enter）
2. 准备好 `.env`：至少有 `OPENROUTER_API_KEY`；本地路径需 Ollama 在跑且已 `ollama pull llama3.2`
3. 改演示格里的题目字符串，分别跑 GPT 与 Llama，对比回答风格


In [ ]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 导入标准库 os：读环境变量（Environment Variables），例如 OpenRouter API Key
import os
# 从 dotenv 导入 load_dotenv：把 .env 文件里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 openai 导入 OpenAI 客户端类：既可打云端，也可打 Ollama 的 OpenAI 兼容接口
from openai import OpenAI
# 从 IPython.display 导入展示工具：Markdown 渲染、首次 display、流式 update_display
from IPython.display import Markdown, display, update_display


In [ ]:
# ========== 环境 + 模型常量：两个后端共用同一套 OpenAI SDK ==========

# 加载 .env；override=True 表示用文件里的值覆盖进程里已有同名环境变量
load_dotenv(override=True)
# 云端小模型名 + 本地 Ollama 模型名（字符串必须和实际可用模型一致）
MODEL_GPT, MODEL_LLAMA = 'gpt-4o-mini', 'llama3.2'

# 每个模型对应：base_url（API 根地址）与 api_key（Ollama 常用占位 "ollama"）
MODEL_CONFIG = {
    MODEL_LLAMA: {"base_url": "http://localhost:11434/v1", "api_key": "ollama"},
    MODEL_GPT: {"base_url": "https://openrouter.ai/api/v1", "api_key": os.getenv("OPENROUTER_API_KEY")},
}

# 启动时验证一次 OpenRouter 密钥（Ollama 不需要真实密钥）
_key = MODEL_CONFIG[MODEL_GPT]["api_key"]
# 缺失或过短时打印英文提示（影响排查的文案保持原文，不翻译）
if not (_key and len(_key) > 10):
    print("OpenRouter API key may be missing. Check .env and troubleshooting notebook.")


In [ ]:
# ========== 工厂函数：按模型名取出配置，构造 OpenAI 客户端 ==========

# model：传 MODEL_GPT 或 MODEL_LLAMA，决定走哪条 base_url
def get_client(model):
    # 从字典取配置；未知模型名时 get 返回 None
    cfg = MODEL_CONFIG.get(model)
    # 配置不存在 → 抛 ValueError（错误文案保持英文，便于和原逻辑对照）
    if not cfg:
        raise ValueError(f"Unknown model: {model}. Use {MODEL_GPT} or {MODEL_LLAMA}")
    # 用对应 base_url / api_key 创建客户端（同一套 SDK，两个后端）
    return OpenAI(base_url=cfg["base_url"], api_key=cfg["api_key"])


In [ ]:
# ========== 下一格：系统提示词、用户模板、示例题与拼装函数 ==========


In [ ]:
# ========== 提示词：系统设计面试专家（发给模型的英文正文勿改译） ==========

# system prompt：定角色、五步讨论结构、输出风格与语气（改译会改变模型行为）
SYSTEM_PROMPT = """
You are a senior staff engineer conducting a system design interview at a top tech company (FAANG-level). Your role is to guide candidates through a structured, realistic system design discussion.

## 你的方法

1. **Requirements Clarification** — Start by clarifying functional and non-functional requirements. Ask about scale (DAU, QPS, storage), consistency needs, latency targets, and key use cases. Make reasonable assumptions when the user doesn't specify.

2. **High-Level Design** — Propose a top-level architecture: clients, load balancers, API servers, core services, databases, caches, message queues. Draw ASCII diagrams when helpful. Identify the main components and data flow.

3. **Deep Dive** — Zoom into 2-3 critical components: data models, sharding strategy, caching layers, replication, or consistency mechanisms. Discuss trade-offs (e.g., consistency vs availability, read vs write optimization).

4. **Scale & Bottlenecks** — Address scalability: horizontal vs vertical scaling, back-of-envelope capacity estimates (storage, bandwidth, QPS). Identify potential bottlenecks and mitigation strategies.

5. **Fault Tolerance & Operations** — Briefly cover failure modes, replication, failover, monitoring, and operational concerns.

## 输出风格

- Use clear markdown: headers, bullet points, code blocks for schemas or configs.
- Include simple ASCII diagrams for architecture (e.g., Client → LB → API → DB).
- Be concise but thorough. Prioritize clarity over length.
- When making assumptions, state them explicitly (e.g., "Assuming 10M DAU...").
- Reference real-world patterns: consistent hashing, write-ahead logs, leader election, etc.

#＃ 语气

- Professional and interview-like. Assume the "candidate" (user) is competent and engaged.
- Don't over-explain basics; focus on the non-obvious and trade-off discussions.
"""

# 用户侧模板：{question} 是题目；{optional_context} 可放规模/约束（可为空）
USER_PROMPT_TEMPLATE = """
Design {question}

{optional_context}
"""

# 可替换进演示格的示例系统设计题（英文题目保持原样，便于模型理解常见题库）
EXAMPLE_QUESTIONS = [
    "Design a URL shortener like bit.ly",
    "Design a rate limiter for an API",
    "Design a distributed cache (like Redis)",
    "Design a chat system (like Slack or WhatsApp)",
    "Design YouTube or Netflix (video streaming)",
    "Design a search autocomplete system",
    "Design a notification system",
]


# 把题目 + 可选上下文填进 USER_PROMPT_TEMPLATE
def build_user_prompt(question: str, context: str = "") -> str:
    # 有非空白 context 时加 "Context/Constraints:" 前缀；否则 optional_context 为空串
    ctx = f"Context/Constraints:\n{context}\n\n" if context.strip() else ""
    # format 填入 question 与 optional_context
    return USER_PROMPT_TEMPLATE.format(question=question, optional_context=ctx)


In [ ]:
# ========== 主函数：组装 messages，流式或一次性拿回答 ==========

# question：题目；model：后端；context：规模/约束；stream：默认 True 边生成边刷屏
def ask_system_design(question: str, model: str = MODEL_GPT, context: str = "", stream: bool = True):
    """Ask a system design question. Streams by default."""
    # 按模型名拿到对应 OpenAI 客户端（OpenRouter 或本机 Ollama）
    client = get_client(model)
    # messages：system 定面试官角色；user 由 build_user_prompt 拼出
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(question, context)},
    ]
    # Chat Completions；stream= 与函数参数一致
    response = client.chat.completions.create(model=model, messages=messages, stream=stream)
    # 先占一个可更新的空 Markdown 显示位（display_id 用于后续刷新）
    display_handle = display(Markdown(""), display_id=True)
    # 非流式：直接返回完整 content
    if not stream:
        return response.choices[0].message.content
    # 流式：累积全文，每来一块就 update_display
    full = ""
    for chunk in response:
        # delta.content 可能是 None（例如结束块），用 or "" 兜底
        content = chunk.choices[0].delta.content or ""
        full += content
        update_display(Markdown(full), display_id=display_handle.display_id)
    # 返回完整字符串，方便后续再处理
    return full


In [ ]:
# ========== 演示 A：云端 gpt-4o-mini（经 OpenRouter）流式回答 ==========

# 题目字符串保持英文（发给模型的可运行内容不翻译）
ask_system_design("Design a URL shortener like bit.ly", model=MODEL_GPT)


In [ ]:
# ========== 演示 B：本地 Llama 3.2（Ollama OpenAI 兼容口）流式回答 ==========

# 同一套 ask_system_design，只把 model 换成 MODEL_LLAMA，对比云端 vs 本地
ask_system_design("Design a rate limiter for an API", model=MODEL_LLAMA)


In [ ]:
# ========== 演示 C：带可选上下文（规模、约束）的提问 ==========

# context 会拼进 user prompt 的 Context/Constraints 段，引导模型按指定规模讨论
ask_system_design(
    "Design a chat system like Slack",
    model=MODEL_GPT,
    context="Scale: 50M DAU, 10B messages/day. Focus on real-time delivery and message ordering."
)
